# AI-Powered Quiz Application
### Problem Statement #4 - AI Internship Training Project

**Features:** Document Upload -> Question Generation -> Answer Evaluation -> Result Summary

**Stack:** PyMuPDF | transformers | sentence-transformers | Gradio

---

In [1]:
# Step 1: Install all required open-source packages
!pip install -q gradio pymupdf sentence-transformers transformers torch accelerate
!pip install -q python-docx pandas numpy nltk
print('All packages installed!')

All packages installed!


In [2]:
# Step 2: Imports
import os, re, random
from pathlib import Path
import fitz
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import gradio as gr
print('All imports successful!')

All imports successful!


In [3]:
# Step 3: Document Ingestion Module
class DocumentIngester:
    def extract_pdf(self, fp):
        doc = fitz.open(fp)
        txt = ''.join(p.get_text('text') + chr(10) for p in doc)
        doc.close()
        return txt.strip()

    def extract_txt(self, fp):
        with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read().strip()

    def extract_docx(self, fp):
        from docx import Document
        d = Document(fp)
        return chr(10).join(p.text for p in d.paragraphs if p.text.strip())

    def extract(self, fp):
        ext = Path(fp).suffix.lower()
        if ext == '.pdf':   return self.extract_pdf(fp)
        if ext == '.txt':   return self.extract_txt(fp)
        if ext in ('.docx', '.doc'): return self.extract_docx(fp)
        raise ValueError(f'Unsupported: {ext}')

    def chunk_text(self, text, chunk_size=300):
        sentences = sent_tokenize(text)
        chunks, cur, cur_len = [], [], 0
        for s in sentences:
            wlen = len(s.split())
            if cur_len + wlen > chunk_size and cur:
                chunks.append(' '.join(cur))
                cur = cur[-2:]
                cur_len = sum(len(x.split()) for x in cur)
            cur.append(s)
            cur_len += wlen
        if cur: chunks.append(' '.join(cur))
        return [c for c in chunks if len(c.split()) > 30]

ingester = DocumentIngester()
print('DocumentIngester ready!')

DocumentIngester ready!


In [4]:
# Step 4: Load Question Generation Model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print('Loading QG model and tokenizer... (may take 1-2 min first time)')

tokenizer = AutoTokenizer.from_pretrained('mrm8488/t5-base-finetuned-question-generation-ap')
model = AutoModelForSeq2SeqLM.from_pretrained('mrm8488/t5-base-finetuned-question-generation-ap')

# Create a function that mimics the pipeline's call signature
def custom_qg_pipeline(inputs, max_length=128, num_beams=4, device=-1):
    # The device argument can be handled by moving the model to GPU if device != -1
    # For this setup, we'll assume the model is already on the correct device if device is specified
    # Or, we can move it here if device != -1
    input_ids = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True).input_ids

    # Move input_ids to the specified device if applicable
    if device != -1:
        # Ensure the model is also on the correct device, e.g., model.to(f'cuda:{device}')
        # For simplicity in this Colab context, we might not need explicit device management if default is GPU
        if hasattr(model, 'device') and model.device.type == 'cpu' and device >= 0:
             # Only move if model is on CPU and a GPU device is requested
            model.to(device)
        input_ids = input_ids.to(model.device) # Ensure inputs are on same device as model

    outputs = model.generate(input_ids, max_length=max_length, num_beams=num_beams, early_stopping=True)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{'generated_text': generated_text}]

qg_pipe = custom_qg_pipeline
print('QG model loaded!')

Loading QG model and tokenizer... (may take 1-2 min first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


QG model loaded!


In [5]:
# Step 5: Question Generator
class QuestionGenerator:
    def __init__(self, qg):
        self.qg = qg

    def gen_q(self, answer_text, context):
        inp = 'answer: ' + answer_text + '  context: ' + context
        try:
            r = self.qg(inp, max_length=128, num_beams=4)[0]['generated_text']
            r = r.replace('question:', '').strip()
            if r and r[-1] not in '.?!': r += '?'
            return r
        except:
            return None

    def key_answer(self, chunk):
        sents = sent_tokenize(chunk)
        for s in sents:
            if len(s.split()) > 8: return s
        return sents[0] if sents else chunk[:200]

    def generate_quiz(self, chunks, n=5):
        n = min(n, len(chunks))
        quiz = []
        for i, chunk in enumerate(random.sample(chunks, n)):
            key_ans = self.key_answer(chunk)
            q = self.gen_q(key_ans, chunk)
            if q:
                quiz.append({'id': i+1, 'question': q, 'context': chunk,
                             'reference_answer': key_ans})
        return quiz

qgen = QuestionGenerator(qg_pipe)
print('QuestionGenerator ready!')

QuestionGenerator ready!


In [6]:
# Step 6: Load Embedding Model
print('Loading semantic similarity model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded!')

Loading semantic similarity model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded!


In [7]:
# Step 7: Answer Evaluator
class AnswerEvaluator:
    def __init__(self, model):
        self.model = model

    def sem_score(self, ua, ra):
        if not ua.strip(): return 0.0
        eu = self.model.encode(ua, convert_to_tensor=True)
        er = self.model.encode(ra, convert_to_tensor=True)
        return max(0.0, util.cos_sim(eu, er).item())

    def kw_score(self, ua, ra):
        if not ua.strip(): return 0.0
        stop = {'the','a','an','is','are','was','were','to','of','and',
                'in','it','that','this','with','for','on','at','by','from'}
        ref = set(w.lower() for w in re.findall(r'\b\w+\b', ra)
                  if w.lower() not in stop and len(w) > 3)
        usr = set(w.lower() for w in re.findall(r'\b\w+\b', ua))
        return len(ref & usr) / len(ref) if ref else 0.5

    def evaluate(self, ua, ra, ctx):
        sem = self.sem_score(ua, ra)
        kw  = self.kw_score(ua, ra)
        score = round((sem * 0.65 + kw * 0.35) * 10, 2)
        if score >= 8:   grade, color = 'Excellent', '#22c55e'
        elif score >= 6: grade, color = 'Good', '#84cc16'
        elif score >= 4: grade, color = 'Partial', '#f59e0b'
        else:            grade, color = 'Needs Improvement', '#ef4444'
        if sem < 0.3:          tip = 'Off-topic. Re-read the context carefully.'
        elif kw < 0.3:         tip = 'Right direction but missing key terms.'
        elif sem >= 0.7:       tip = 'Excellent! Captures both meaning and key concepts.'
        else:                  tip = 'Good understanding. Try to be more specific.'
        return {'score': score, 'grade': grade, 'grade_color': color,
                'semantic_similarity': round(sem*100, 1),
                'keyword_overlap': round(kw*100, 1),
                'feedback': tip, 'reference_answer': ra}

evaluator = AnswerEvaluator(embed_model)
print('AnswerEvaluator ready!')

AnswerEvaluator ready!


In [8]:
# Step 8: Quiz Session Manager
class QuizSession:
    def __init__(self):
        self.reset()

    def reset(self):
        self.quiz_items = []
        self.current_index = 0
        self.results = []

    def load_document(self, fp, num_questions=5):
        self.reset()
        text = ingester.extract(fp)
        chunks = ingester.chunk_text(text)
        if not chunks: raise ValueError('Document too short or unreadable.')
        self.quiz_items = qgen.generate_quiz(chunks, n=num_questions)
        return len(self.quiz_items)

    def current_question(self):
        if self.current_index < len(self.quiz_items):
            return self.quiz_items[self.current_index]
        return None

    def submit_answer(self, ua):
        item = self.current_question()
        if not item: return None
        r = evaluator.evaluate(ua, item['reference_answer'], item['context'])
        r.update({'question': item['question'], 'user_answer': ua, 'q_num': item['id']})
        self.results.append(r)
        self.current_index += 1
        return r

    def is_complete(self):
        return self.current_index >= len(self.quiz_items)

    def summary(self):
        if not self.results: return {}
        scores = [r['score'] for r in self.results]
        avg = round(sum(scores)/len(scores), 2)
        return {'total_questions': len(self.results), 'average_score': avg,
                'percentage': round(avg*10, 1), 'highest': max(scores),
                'lowest': min(scores), 'results': self.results}

    def export_results(self):
        rows = [{'Q#': r['q_num'], 'Question': r['question'],
                 'Your Answer': r['user_answer'], 'Reference': r['reference_answer'],
                 'Score/10': r['score'], 'Grade': r['grade'],
                 'Semantic%': r['semantic_similarity'], 'Keyword%': r['keyword_overlap'],
                 'Feedback': r['feedback']} for r in self.results]
        return pd.DataFrame(rows)

session = QuizSession()
print('QuizSession ready!')

QuizSession ready!


In [9]:
def q_html(item, q_num, total):
    pct = int((q_num-1)/total*100)
    qtext = item['question']
    ctx = item['context'][:250]
    return f"""
    <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #eee; background-color: #1e1e1e; padding: 20px; border-radius: 12px; border: 1px solid #333;">
        <div style="display:flex;justify-content:space-between;margin-bottom:10px; align-items: center;">
            <span style="color:#00bcd4;font-weight:700; font-size: 1.1rem;">QUESTION {q_num} of {total}</span>
            <span style="color:#666; font-size: 0.9rem;">{pct}% complete</span>
        </div>
        <div style="background:#333;border-radius:4px;height:6px;margin-bottom:16px;">
            <div style="background:linear-gradient(90deg,#00bcd4,#00e676);height:100%;width:{pct}%;border-radius:4px;"></div>
        </div>
        <div style="background-color:#2a2a2a; border:1px solid #444; border-radius:10px; padding:18px; margin-bottom:16px;">
            <p style="font-size:1.15rem;font-weight:600;color:#f1f1f1;margin:0;">{qtext}</p>
        </div>
        <div style="border-left:4px solid #00bcd4; padding:10px 15px; background-color:#252525; border-radius:0 8px 8px 0;">
            <p style="font-size:0.75rem;color:#999;margin:0 0 5px 0;font-weight:700;">CONTEXT HINT</p>
            <p style="font-size:0.85rem;color:#ccc;margin:0;">{ctx}...</p>
        </div>
    </div>
    """

def r_html(r):
    return f"""
    <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #eee; background-color: #1e1e1e; padding: 20px; border-radius: 12px; border: 1px solid #333;">
        <div style="display:flex;align-items:center;gap:16px;margin-bottom:20px;">
            <div style="background:linear-gradient(135deg,#00bcd4,#00e676);width:60px;height:60px;
            border-radius:50%;display:flex;align-items:center;justify-content:center;
            font-size:1.5rem;font-weight:800;color:white;">{r['score']}</div>
            <div>
                <div style="font-size:1.2rem;font-weight:700;color:#f1f1f1;">{r['grade']}</div>
                <div style="font-size:0.8rem;color:#888;">out of 10 points</div>
            </div>
        </div>
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:16px;">
            <div style="background-color:#2a2a2a; border:1px solid #444; border-radius:10px; padding:14px;">
                <div style="color:#999;font-size:0.75rem;font-weight:700;">SEMANTIC MATCH</div>
                <div style="font-size:1.3rem;font-weight:700;color:#00bcd4;">{r['semantic_similarity']}%</div>
            </div>
            <div style="background-color:#2a2a2a; border:1px solid #444; border-radius:10px; padding:14px;">
                <div style="color:#999;font-size:0.75rem;font-weight:700;">KEYWORD MATCH</div>
                <div style="font-size:1.3rem;font-weight:700;color:#00e676;">{r['keyword_overlap']}%</div>
            </div>
        </div>
        <div style="background-color:#252525; border:1px solid #444; border-radius:10px; padding:14px; margin-bottom:12px;">
            <div style="color:#999;font-size:0.75rem;font-weight:700;margin-bottom:8px;">FEEDBACK</div>
            <p style="margin:0;color:#ccc;font-size:0.9rem;">{r['feedback']}</p>
        </div>
        <div style="background-color:#252525; border:1px solid #444; border-radius:10px; padding:14px;">
            <div style="color:#00bcd4;font-size:0.75rem;font-weight:700;margin-bottom:8px;">REFERENCE ANSWER</div>
            <p style="margin:0;color:#ccc;font-size:0.88rem;">{r['reference_answer']}</p>
        </div>
    </div>
    """

def s_html(summary):
    pct = summary['percentage']
    if pct >= 80:   lbl, col = 'Outstanding!', '#00e676'
    elif pct >= 60: lbl, col = 'Well Done!', '#aeea00'
    elif pct >= 40: lbl, col = 'Keep Practicing', '#ffb300'
    else:           lbl, col = 'More Study Needed', '#ff3d00'

    rows_html = ""
    for r in summary['results']:
        gc = r['grade_color']
        rows_html += f"""
            <div style="display:flex;gap:12px;padding:12px 0;border-bottom:1px solid #2a2a2a;align-items:flex-start;">
                <div style="background:{gc}33;color:{col};border:1px solid {gc}55;width:38px;height:38px;
                border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1rem;">{r['score']}</div>
                <div>
                    <p style="margin:0 0 4px 0;color:#eee;font-size:0.9rem;">{r['question']}</p>
                    <p style="margin:0;color:#999;font-size:0.78rem;">{r['grade']} |
                    Semantic: {r['semantic_similarity']}% | Keywords: {r['keyword_overlap']}%</p>
                </div>
            </div>
            """
    return f"""
    <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;color:#eee;">
        <div style="background-color:#1e1e1e; border:1px solid #333;
        border-radius:12px;padding:28px;text-align:center;margin-bottom:20px;">
            <div style="font-size:3rem;font-weight:800;color:{col};">{pct}%</div>
            <div style="font-size:1.2rem;color:#00bcd4;font-weight:700;margin-top:5px;">{lbl}</div>
            <div style="color:#999;font-size:0.85rem;margin-top:10px;">
                Total Questions: {summary['total_questions']} | Average: {summary['average_score']}/10 |
                Highest: {summary['highest']} | Lowest: {summary['lowest']}
            </div>
        </div>
        <div style="background-color:#1e1e1e;border:1px solid #333;border-radius:12px;padding:20px;">
            <h3 style="margin:0 0 12px 0;font-size:0.9rem;color:#00bcd4;text-transform:uppercase;">DETAILED RESULTS</h3>
            {rows_html}
        </div>
    </div>
    """

print('HTML helpers ready!')

HTML helpers ready!


In [10]:
# Step 10: Event Handlers
state = {'loaded': False}

def handle_upload(file, num_q):
    if file is None:
        return ('<p style="color:#ef4444;">Please upload a document first.</p>',
                gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False), '', gr.update(visible=False))
    try:
        n = session.load_document(file.name, num_questions=int(num_q))
        state['loaded'] = True
        item = session.current_question()
        return (q_html(item, 1, n), gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=False), '', gr.update(visible=False))
    except Exception as e:
        return (f'<p style="color:#ef4444;">Error: {str(e)}</p>',
                gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False), '', gr.update(visible=False))

def handle_submit(ua):
    if not state.get('loaded'):
        return ('<p style="color:#f59e0b;">Upload a document first.</p>',
                gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False), '', gr.update(visible=False))
    if not ua.strip():
        return ('<p style="color:#f59e0b;">Please enter an answer first.</p>',
                gr.update(visible=True), gr.update(visible=True),
                gr.update(visible=False), ua, gr.update(visible=False))
    result = session.submit_answer(ua)
    rh = r_html(result)
    if session.is_complete():
        return (rh, gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=True), '', gr.update(visible=True, value='See Final Summary'))
    return (rh, gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=True), '', gr.update(visible=True, value='Next Question'))

def handle_next(btn_lbl):
    if 'Summary' in str(btn_lbl) or session.is_complete():
        sm = session.summary()
        return ('', gr.update(visible=False), gr.update(visible=False),
                gr.update(visible=False), '', gr.update(visible=False),
                gr.update(visible=True, value=s_html(sm)))
    item = session.current_question()
    qn = session.current_index + 1
    total = len(session.quiz_items)
    return (q_html(item, qn, total), gr.update(visible=True), gr.update(visible=True),
            gr.update(visible=False), '', gr.update(visible=False), gr.update(visible=False))

def handle_restart():
    session.reset()
    state['loaded'] = False
    return ('<p style="color:#64748b;text-align:center;padding:40px;">Upload a document to begin.</p>',
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), '', gr.update(visible=False),
            gr.update(visible=False, value=''))

def export_handler():
    df = session.export_results()
    path = '/content/quiz_results.csv'
    df.to_csv(path, index=False)
    return f'Exported {len(df)} results to quiz_results.csv', gr.update(visible=True, value=path)

print('Event handlers ready!')

Event handlers ready!


In [11]:
HEADER = '''
<div style="background-color: #1a1a1a; border: 1px solid #333; border-radius: 12px; padding: 24px; text-align: center; margin-bottom: 16px;">
    <div style="font-size: 2.2rem; font-weight: 700; color: #00bcd4; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
        AI QuizMaster
    </div>
    <p style="color: #bbb; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 8px 0 16px;">
        Upload your document, get AI-generated questions, and evaluate your answers with smart scoring.
    </p>
    <div style="display: flex; justify-content: center; gap: 10px; flex-wrap: wrap;">
        <span style="background-color: #2a2a2a; border: 1px solid #444; padding: 4px 10px; border-radius: 16px; color: #00bcd4; font-size: 0.75rem; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">Multi-Format Docs</span>
        <span style="background-color: #2a2a2a; border: 1px solid #444; padding: 4px 10px; border-radius: 16px; color: #00bcd4; font-size: 0.75rem; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">T5 Question Gen</span>
        <span style="background-color: #2a2a2a; border: 1px solid #444; padding: 4px 10px; border-radius: 16px; color: #00bcd4; font-size: 0.75rem; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">Semantic Scoring</span>
        <span style="background-color: #2a2a2a; border: 1px solid #444; padding: 4px 10px; border-radius: 16px; color: #00bcd4; font-size: 0.75rem; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">Open Source Stack</span>
    </div>
</div>
'''

HOW = '''
<div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-size: 0.8rem; color: #999; margin-top: 16px; padding: 10px; border-radius: 8px; background-color: #1a1a1a;">
    <p style="font-weight: 600; color: #00bcd4; margin-bottom: 8px;">HOW IT WORKS</p>
    <ul style="list-style-type: disc; padding-left: 20px; margin: 0;">
        <li style="margin-bottom: 5px;">Upload your study document (PDF, TXT, DOCX).</li>
        <li style="margin-bottom: 5px;">AI intelligently generates questions from your content.</li>
        <li style="margin-bottom: 5px;">Type your answers in the provided box.</li>
        <li style="margin-bottom: 5px;">LLM evaluates your response using advanced semantic matching.</li>
        <li style="margin-bottom: 5px;">Receive detailed, per-question feedback.</li>
        <li>Review your overall performance and export a CSV report.</li>
    </ul>
</div>
'''

INIT = '<p style="color: #aaa; text-align: center; padding: 50px; font-family: \'Segoe UI\', Tahoma, Geneva, Verdana, sans-serif; font-size: 1.1rem;">Ready to learn? Upload a document and click "Generate Quiz" to begin.</p>'

with gr.Blocks(title='AI Quiz App') as app:
    gr.HTML(HEADER)
    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML('<h3 style="color:#00bcd4;font-family:\'Segoe UI\', Tahoma, Geneva, Verdana, sans-serif;">Document Setup</h3>')
            file_input = gr.File(label='Upload (PDF / TXT / DOCX)', file_types=['.pdf','.txt','.docx'])
            num_q = gr.Slider(minimum=3, maximum=10, value=5, step=1, label='Number of Questions')
            upload_btn  = gr.Button('Generate Quiz', variant='primary')
            restart_btn = gr.Button('Start Over', variant='secondary')
            gr.HTML(HOW)
        with gr.Column(scale=2):
            question_html = gr.HTML(value=INIT)
            answer_box     = gr.Textbox(label='Your Answer', placeholder='Type your answer here...', lines=4, visible=False)
            submit_btn    = gr.Button('Submit Answer', variant='primary', visible=False)
            result_html   = gr.HTML(visible=False)
            next_btn      = gr.Button('Next Question', variant='primary', visible=False)
    summary_html = gr.HTML(visible=False)
    with gr.Row():
        export_btn    = gr.Button('Export Results as CSV')
        export_status = gr.Textbox(label='Export Status', interactive=False)
    export_file = gr.File(label='Download Results', visible=False)

    upload_btn.click(
        fn=handle_upload, inputs=[file_input, num_q],
        outputs=[question_html, answer_box, submit_btn, result_html, answer_box, next_btn])
    submit_btn.click(
        fn=handle_submit, inputs=[answer_box],
        outputs=[result_html, answer_box, submit_btn, result_html, answer_box, next_btn])
    next_btn.click(
        fn=handle_next, inputs=[next_btn],
        outputs=[question_html, answer_box, submit_btn, result_html, answer_box, next_btn, summary_html])
    restart_btn.click(
        fn=handle_restart, inputs=[],
        outputs=[question_html, answer_box, submit_btn, result_html, answer_box, next_btn, summary_html])
    export_btn.click(
        fn=export_handler, inputs=[], outputs=[export_status, export_file])

print('Launching app...')
app.launch(share=True, debug=False)

Launching app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e3c0fe56c4a67c284.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
# Quick pipeline test (run this cell to verify the pipeline works without UI)
sample = (
    'Machine learning enables computers to learn from data without explicit programming. '
    'Supervised learning trains models on labeled datasets. '
    'Unsupervised learning finds hidden patterns without labels. '
    'Deep learning uses neural networks with many layers for complex pattern modeling. '
    'Natural language processing allows machines to understand human language. '
    'Reinforcement learning trains agents using reward and punishment signals. '
    'Transfer learning reuses pre-trained models on new tasks efficiently.'
)
print('Testing pipeline...\n')
chunks = ingester.chunk_text(sample, chunk_size=60)
quiz   = qgen.generate_quiz(chunks, n=2)
print(f'Generated {len(quiz)} questions\n')
for item in quiz:
    print(f'Q{item["id"]}: {item["question"]}')
    res = evaluator.evaluate(
        'Deep learning uses neural networks with many layers.',
        item['reference_answer'], item['context'])
    print(f'  Score: {res["score"]}/10 | Grade: {res["grade"]}')
    print(f'  Semantic: {res["semantic_similarity"]}% | Keywords: {res["keyword_overlap"]}%')
    print(f'  Feedback: {res["feedback"]}\n')

Testing pipeline...

Generated 1 questions

Q1: What does machine learning do?
  Score: 3.04/10 | Grade: Needs Improvement
  Semantic: 40.7% | Keywords: 11.1%
  Feedback: Right direction but missing key terms.



---
## Architecture

```
+------------------------------------------------------------+
|              AI Quiz Application (PS #4)                   |
+---------------+-------------+-------------+---------------+
| Module 1      | Module 2    | Module 3    | Module 4      |
| Document      | Question    | Answer      | Result        |
| Ingestion     | Generation  | Evaluation  | Summary       |
+---------------+-------------+-------------+---------------+
| PyMuPDF       | T5-base QG  | MiniLM-L6   | pandas CSV    |
| python-docx   | SQuAD FT    | Cosine Sim  | Gradio UI     |
| NLTK chunking | num_beams=4 | KW overlap  | HTML Report   |
+---------------+-------------+-------------+---------------+
                    All 100% Open Source
```

**Scoring:** `Score (0-10) = Semantic x 0.65 + Keyword x 0.35`

| Score | Grade             |
|-------|-------------------|
| 8-10  | Excellent         |
| 6-8   | Good              |
| 4-6   | Partial           |
| 0-4   | Needs Improvement |